<a href="https://colab.research.google.com/github/hamzaqarni1/DeepLearning/blob/main/Tutorial_14/Tutorial_14_B_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from google.colab import files
import random

# --- Set Seed for Reproducibility ---
torch.manual_seed(42)
np.random.seed(42)

print("Starting Tutorial 14_B Upgraded Execution Pipeline...")

# ==============================================================================
# PART 1: TASKS 1 & 2 - CUSTOM TRANSLATION (MANY-TO-MANY) & HYPERPARAMETERS
# ==============================================================================

english_sentences = [
    "hello how are you",
    "i am fine thank you",
    "what is your name",
    "i love programming",
    "deep learning is fun"
]
french_sentences = [
    "bonjour comment allez vous",
    "je vais bien merci",
    "comment vous appelez vous",
    "jaime la programmation",
    "lapprentissage profond est amusant"
]

def build_vocab(sentences):
    vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
    for sentence in sentences:
        for word in sentence.split():
            if word not in vocab: vocab[word] = len(vocab)
    return vocab, {v: k for k, v in vocab.items()}

eng_vocab, idx2eng = build_vocab(english_sentences)
fra_vocab, idx2fra = build_vocab(french_sentences)

def encode_sentence(sentence, vocab):
    return [vocab["<SOS>"]] + [vocab.get(w, vocab["<UNK>"]) for w in sentence.split()] + [vocab["<EOS>"]]

X_encoded = [encode_sentence(s, eng_vocab) for s in english_sentences]
y_encoded = [encode_sentence(s, fra_vocab) for s in french_sentences]

max_len_eng = max(len(s) for s in X_encoded)
max_len_fra = max(len(s) for s in y_encoded)

X_padded = [s + [eng_vocab["<PAD>"]] * (max_len_eng - len(s)) for s in X_encoded]
y_padded = [s + [fra_vocab["<PAD>"]] * (max_len_fra - len(s)) for s in y_encoded]

X_tensor = torch.tensor(X_padded, dtype=torch.long)
y_tensor = torch.tensor(y_padded, dtype=torch.long)

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        _, hidden = self.rnn(self.embedding(x))
        return hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        out, hidden = self.rnn(self.embedding(x), hidden)
        return self.fc(out), hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        hidden = self.encoder(src)
        output, _ = self.decoder(trg[:, :-1], hidden)
        return output

def train_translation_model(hidden_dim, lr, epochs, name):
    print(f"\nTraining {name} Model (Units={hidden_dim}, LR={lr}, Epochs={epochs})...")
    encoder = Encoder(len(eng_vocab), 64, hidden_dim)
    decoder = Decoder(len(fra_vocab), 64, hidden_dim)
    model = Seq2Seq(encoder, decoder)

    criterion = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    loss_hist = []
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        output = model(X_tensor, y_tensor)
        loss = criterion(output.reshape(-1, len(fra_vocab)), y_tensor[:, 1:].reshape(-1))
        loss.backward()
        optimizer.step()
        loss_hist.append(loss.item())

        if (epoch + 1) % max(1, epochs // 5) == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")
    return model, loss_hist

configs = [
    {"name": "Baseline", "units": 32, "lr": 0.001, "epochs": 50, "color": "blue"},
    {"name": "Moderate", "units": 64, "lr": 0.005, "epochs": 100, "color": "orange"},
    {"name": "Aggressive", "units": 128, "lr": 0.01, "epochs": 150, "color": "red"}
]

models, histories, final_losses = {}, {}, []
for cfg in configs:
    m, h = train_translation_model(cfg["units"], cfg["lr"], cfg["epochs"], cfg["name"])
    models[cfg["name"]] = m
    histories[cfg["name"]] = h
    final_losses.append(h[-1])

best_model = models["Aggressive"]

# --- Visual 1: Multi-Model Translation Loss Curve ---
plt.figure(figsize=(10, 5))
for cfg in configs:
    plt.plot(range(1, cfg["epochs"]+1), histories[cfg["name"]], label=f"{cfg['name']} (U:{cfg['units']}, LR:{cfg['lr']})", color=cfg["color"], linewidth=2)
plt.title("Task 2: Seq2Seq Translation Hyperparameter Optimization", fontweight='bold')
plt.xlabel("Epochs")
plt.ylabel("CrossEntropy Loss")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("trans_multi_loss.png", dpi=300)
plt.close(); files.download("trans_multi_loss.png")

# --- Visual 2: Final Loss Bar Chart ---
plt.figure(figsize=(8, 4))
bars = plt.bar([c["name"] for c in configs], final_losses, color=[c["color"] for c in configs], alpha=0.8)
plt.title("Terminal Loss by Model Configuration", fontweight='bold')
plt.ylabel("Final CrossEntropy Loss")
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2.0, bar.get_height() + 0.01, f'{bar.get_height():.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig("trans_final_loss_bar.png", dpi=300)
plt.close(); files.download("trans_final_loss_bar.png")

# --- Visual 3: Hyperparameter Matrix ---
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.axis('tight'); ax.axis('off')
summary_data = [[c['name'], c['units'], c['epochs'], c['lr'], f"{f:.4f}"] for c, f in zip(configs, final_losses)]
table = ax.table(cellText=summary_data, colLabels=['Config', 'Hidden Units', 'Epochs', 'Learning Rate', 'Final Loss'], loc='center')
table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1, 1.8)
for (row, col), cell in table.get_celld().items():
    if row == 0: cell.set_facecolor('#2c3e50'); cell.set_text_props(weight='bold', color='white')
    elif col == 4 and row > 0:
        cell.set_text_props(weight='bold')
        if "Aggressive" in summary_data[row-1][0]: cell.set_facecolor('#d4edda')
plt.title("Hyperparameter Matrix", fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("trans_hyperparams.png", dpi=300, bbox_inches='tight')
plt.close(); files.download("trans_hyperparams.png")

# --- Visual 4: PCA Word Embeddings ---
embeddings = best_model.encoder.embedding.weight.detach().numpy()
embeddings_2d = PCA(n_components=2).fit_transform(embeddings)
plt.figure(figsize=(9, 6))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], color='mediumseagreen', s=100, edgecolor='black')
for word, idx in eng_vocab.items():
    if word not in ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]:
        plt.annotate(word, (embeddings_2d[idx, 0] + 0.05, embeddings_2d[idx, 1] + 0.05), fontsize=11)
plt.title("PCA Projection of English Source Embeddings", fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("trans_word_pca.png", dpi=300)
plt.close(); files.download("trans_word_pca.png")

# --- Visual 5: Translation Inference Table ---
def translate_sentence(model, sentence):
    model.eval()
    with torch.no_grad():
        encoded = [eng_vocab.get(w, eng_vocab["<UNK>"]) for w in sentence.split()]
        src_tensor = torch.tensor([[eng_vocab["<SOS>"]] + encoded + [eng_vocab["<EOS>"]]], dtype=torch.long)
        hidden = model.encoder(src_tensor)
        trg_indexes = [fra_vocab["<SOS>"]]
        for _ in range(10):
            out, hidden = model.decoder(torch.tensor([[trg_indexes[-1]]], dtype=torch.long), hidden)
            pred_token = out.argmax(2).item()
            if pred_token == fra_vocab["<EOS>"]: break
            trg_indexes.append(pred_token)
    return " ".join([idx2fra[i] for i in trg_indexes[1:]])

test_sentences = ["what is your name", "deep learning is fun"]
results = [[s, translate_sentence(best_model, s)] for s in test_sentences]
fig, ax = plt.subplots(figsize=(9, 2))
ax.axis('tight'); ax.axis('off')
table = ax.table(cellText=results, colLabels=['English Input (Custom)', 'French Output (RNN Prediction)'], loc='center')
table.auto_set_font_size(False); table.set_fontsize(12); table.scale(1, 2)
for (row, col), cell in table.get_celld().items():
    if row == 0: cell.set_facecolor('#2c3e50'); cell.set_text_props(weight='bold', color='white')
plt.title("Task 1: Out-of-Distribution Translation Inference", fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("trans_results_table.png", dpi=300, bbox_inches='tight')
plt.close(); files.download("trans_results_table.png")


# ==============================================================================
# PART 2: TASK 3 - ONE-TO-MANY RNN (BABY NAME GENERATOR)
# ==============================================================================
print("\nInitiating Task 3: One-to-Many Baby Name Generator...")

names = ["olivia", "emma", "charlotte", "amelia", "ava", "sophia", "isabella", "mia",
         "liam", "noah", "oliver", "elijah", "james", "william", "benjamin", "lucas"]
char2idx = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2}
for c in set("".join(names)): char2idx[c] = len(char2idx)
idx2char = {v: k for k, v in char2idx.items()}

name_seqs = [[char2idx["<SOS>"]] + [char2idx[c] for c in name] + [char2idx["<EOS>"]] for name in names]
max_name_len = max(len(s) for s in name_seqs)
name_tensor = torch.tensor([s + [char2idx["<PAD>"]] * (max_name_len - len(s)) for s in name_seqs], dtype=torch.long)

class BabyNameGenerator(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(BabyNameGenerator, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        out, hidden = self.rnn(self.embedding(x), hidden)
        return self.fc(out), hidden

name_model = BabyNameGenerator(len(char2idx), 32, 64)
optimizer_name = optim.Adam(name_model.parameters(), lr=0.01)
criterion_name = nn.CrossEntropyLoss(ignore_index=0)

name_loss_hist = []
name_model.train()
for epoch in range(150):
    optimizer_name.zero_grad()
    logits, _ = name_model(name_tensor[:, :-1])
    loss = criterion_name(logits.reshape(-1, len(char2idx)), name_tensor[:, 1:].reshape(-1))
    loss.backward()
    optimizer_name.step()
    name_loss_hist.append(loss.item())

# --- Visual 6: Baby Name Loss Curve ---
plt.figure(figsize=(8, 4.5))
plt.plot(name_loss_hist, color='purple', linewidth=2.5)
plt.title("Task 3: Character-Level RNN Convergence", fontweight='bold')
plt.xlabel("Epochs"); plt.ylabel("CrossEntropy Loss")
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("baby_name_loss.png", dpi=300)
plt.close(); files.download("baby_name_loss.png")

# --- Visual 7: Generated Baby Names Table ---
def generate_name(model, start_char="<SOS>", max_length=10):
    model.eval()
    with torch.no_grad():
        current_char = torch.tensor([[char2idx[start_char]]], dtype=torch.long)
        hidden, generated_name = None, ""
        for _ in range(max_length):
            logits, hidden = model(current_char, hidden)
            probs = torch.softmax(logits.squeeze() / 0.8, dim=0) # Temperature scaling
            next_idx = torch.multinomial(probs, 1).item()
            if next_idx in [char2idx["<EOS>"], char2idx["<PAD>"]]: break
            generated_name += idx2char[next_idx]
            current_char = torch.tensor([[next_idx]], dtype=torch.long)
    return generated_name.capitalize()

generated_names = [[f"Generated Name {i+1}", generate_name(name_model)] for i in range(5)]
fig, ax = plt.subplots(figsize=(6, 3))
ax.axis('tight'); ax.axis('off')
table = ax.table(cellText=generated_names, colLabels=['Entity', 'RNN Generated String'], loc='center')
table.auto_set_font_size(False); table.set_fontsize(12); table.scale(1, 2)
for (row, col), cell in table.get_celld().items():
    if row == 0: cell.set_facecolor('#2c3e50'); cell.set_text_props(weight='bold', color='white')
plt.title("Task 3: One-to-Many Generative Outputs", fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("generated_names_table.png", dpi=300, bbox_inches='tight')
plt.close(); files.download("generated_names_table.png")

print("\nExecution Complete! 7 massive graphical assets have been downloaded for your LaTeX report.")

Starting Tutorial 14_B Upgraded Execution Pipeline...

Training Baseline Model (Units=32, LR=0.001, Epochs=50)...
Epoch 10/50 | Loss: 2.6519
Epoch 20/50 | Loss: 2.2190
Epoch 30/50 | Loss: 1.8422
Epoch 40/50 | Loss: 1.5146
Epoch 50/50 | Loss: 1.2378

Training Moderate Model (Units=64, LR=0.005, Epochs=100)...
Epoch 20/100 | Loss: 0.0664
Epoch 40/100 | Loss: 0.0082
Epoch 60/100 | Loss: 0.0044
Epoch 80/100 | Loss: 0.0034
Epoch 100/100 | Loss: 0.0028

Training Aggressive Model (Units=128, LR=0.01, Epochs=150)...
Epoch 30/150 | Loss: 0.0001
Epoch 60/150 | Loss: 0.0001
Epoch 90/150 | Loss: 0.0000
Epoch 120/150 | Loss: 0.0000
Epoch 150/150 | Loss: 0.0000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Execution Complete! 7 massive graphical assets have been downloaded for your LaTeX report.
